# Análise Financeira ClearBank


Notebook python que lê e valida um arquivo CSV de transações bancárias, agrupa os dados por mês, calcula métricas financeiras, sinaliza movimentações suspeitas e exporta o resultado em JSON.

## Como executar

Rode as células **em ordem**, de cima para baixo (ou `Ambiente de execução → Executar tudo`).
A última célula da seção principal (`Célula de Execução Principal`) encadeia todo o fluxo.

## Saídas geradas

| Arquivo | Conteúdo |
|---|---|
| `relatorio.json` | Relatório completo da análise |
| `grafico.png` | Gráfico de crédito/débito/saldo por mês |


## Fase 1: O arquivo de entrada

O `transacoes.csv` faz parte do repositório. Esta célula o recria **caso ele não exista no repositório**, garantindo que o notebook rode de ponta a ponta no Google Colab sem upload manual.

O arquivo contém propositalmente dados sujos, para exercitar a validação:

- **20 registros válidos** distribuídos em 4 meses (janeiro a abril de 2026)
- **3 transações acima de R$ 10.000,00** (que devem ser sinalizadas como suspeitas)
- **6 registros inválidos**: valor não numérico, `cliente_id` vazio, data fora do padrão, tipo inválido, valor negativo e `id` vazio
- **1 registro duplicado** (mesmo `id` de uma linha já lida)

In [103]:
import os

CSV_CONTEUDO = """id,data,cliente_id,tipo,valor,descricao,categoria
1,2026-01-05,CLI001,credito,3500.00,Salário janeiro,salario
2,2026-01-08,CLI002,credito,4200.00,Salário janeiro,salario
3,2026-01-12,CLI002,debito,180.50,Supermercado,compra
4,2026-01-18,CLI001,debito,320.00,Conta de luz,conta
5,2026-01-25,CLI003,debito,99.90,Streaming,assinatura
6,2026-01-28,CLI003,credito,12500.00,Venda de veículo,transferencia
7,2026-02-03,CLI001,credito,3500.00,Salário fevereiro,salario
8,2026-02-07,CLI002,debito,1450.00,Aluguel,conta
9,2026-02-14,CLI003,credito,15000.00,Transferência recebida,transferencia
10,2026-02-18,CLI002,debito,320.00,Conta de luz,conta
11,2026-02-22,CLI001,debito,89.90,Farmácia,compra
12,2026-03-01,CLI001,credito,3500.00,Salário março,salario
13,2026-03-05,CLI003,debito,2300.00,Reforma da sala,compra
14,2026-03-10,CLI003,debito,99.90,Streaming,assinatura
15,2026-03-16,CLI002,credito,4200.00,Salário março,salario
16,2026-03-21,CLI002,debito,760.25,Mercado do mês,compra
17,2026-04-02,CLI001,credito,3500.00,Salário abril,salario
18,2026-04-09,CLI003,credito,11800.00,Bônus anual,salario
19,2026-04-15,CLI002,debito,1450.00,Aluguel,conta
20,2026-04-27,CLI001,debito,540.75,Material escolar,compra
12,2026-03-01,CLI001,credito,3500.00,Salário março (duplicado),salario
21,2026-04-30,CLI001,debito,abc,Valor não numérico,compra
22,2026-05-03,,debito,200.00,Cliente vazio,transferencia
23,30-05-2026,CLI002,credito,900.00,Data fora do padrão,salario
24,2026-05-11,CLI003,pix,450.00,Tipo inválido,transferencia
25,2026-05-15,CLI001,debito,-75.00,Valor negativo,compra
,2026-05-20,CLI002,debito,120.00,ID vazio,compra
"""

if os.path.exists("transacoes.csv"):
    print("Arquivo 'transacoes.csv' já existe.")
else:
    with open("transacoes.csv", "w", encoding="utf-8") as arquivo:
        arquivo.write(CSV_CONTEUDO)
    print("Arquivo 'transacoes.csv' criado.")

print(f"Quantidade de Registros no arquivo (sem o cabeçalho): {len(CSV_CONTEUDO.strip().splitlines()) - 1}")

Arquivo 'transacoes.csv' já existe.
Quantidade de Registros no arquivo (sem o cabeçalho): 27


## Fase 2. Imports e constantes

Para facilitar a organização, todas as constantes ficam concentradas aqui inclusive o `LIMITE_SUSPEITO` exigido como um dos critérios obrigatórios.
Assim, ajustar uma regra de negócio não exige procurar valores espalhados pelo código.

In [104]:
import csv
import json
from datetime import datetime, date

# --- Configuração ---
ARQUIVO_ENTRADA = "transacoes.csv"
ARQUIVO_SAIDA = "relatorio.json"
ARQUIVO_GRAFICO = "grafico.png"

FORMATO_DATA = "%Y-%m-%d"      # AAAA-MM-DD
FORMATO_MES = "%Y-%m"          # AAAA-MM

TIPOS_VALIDOS = ("credito", "debito")

# Limite para transação suspeita. Qualquer valor acima é acusado
LIMITE_SUSPEITO = 10000.00

print("Imports e constantes carregados.")
print(f"Arquivo de entrada.: {ARQUIVO_ENTRADA}")
print(f"Arquivo de saída...: {ARQUIVO_SAIDA}")
print(f"Limite suspeito....: {LIMITE_SUSPEITO}")

Imports e constantes carregados.
Arquivo de entrada.: transacoes.csv
Arquivo de saída...: relatorio.json
Limite suspeito....: 10000.0


## Fase 3. Funções auxiliares de formatação

`formatar_moeda()` converte um `float` para o padrão monetário brasileiro (`R$ 1.234,56`).

O truque dos três `replace` encadeados existe porque o Python formata no padrão americano (`1,234.56`). Usamos o `X` como marcador temporário para trocar vírgula e ponto de posição sem sobrescrever um caractere que ainda vamos precisar.

In [105]:
def formatar_moeda(valor):
    """Formata um número no padrão monetário brasileiro. Ex.: 3319.5 -> 'R$ 3.319,50'."""
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def formatar_linha(rotulo, valor, largura=15):
    """Alinha rótulo e valor monetário para deixar o relatório legível no terminal."""
    return f"  {rotulo:<{largura}}{formatar_moeda(valor)}"

print("Funções definidas: formatar_moeda(), formatar_linha()")

Funções definidas: formatar_moeda(), formatar_linha()


In [106]:
# --- Teste rápido ---
for teste in [3319.50, 180.5, 15000.0, 1234567.89, 0.0]:
    print(f"{teste:>12} -> {formatar_moeda(teste)}")

      3319.5 -> R$ 3.319,50
       180.5 -> R$ 180,50
     15000.0 -> R$ 15.000,00
  1234567.89 -> R$ 1.234.567,89
         0.0 -> R$ 0,00


## Fase 4. Leitura do arquivo CSV (requisito obrigatório)

Leitura com o módulo `csv` **nativo** (sem pandas), usando `csv.DictReader` (leitura de dicionário) para acessar as colunas pelo nome.

O `with open(...)` garante que o arquivo seja fechado mesmo se ocorrer um erro no meio da leitura. O `try/except FileNotFoundError` (requisito obrigatório) evita que o programa quebre caso o arquivo não exista, se ocorrer devolvemos uma lista vazia e o fluxo segue de forma controlada.

In [107]:
def ler_transacoes(caminho=ARQUIVO_ENTRADA):
    """Lê o CSV e devolve a lista de transações brutas (dicionários), sem validar.
    Retorna uma lista vazia se o arquivo não existir ou não puder ser lido.
    """
    linhas = []
    try:
        with open(caminho, mode="r", encoding="utf-8", newline="") as arquivo:
            leitor = csv.DictReader(arquivo)
            for linha in leitor:
                linhas.append(linha)
    except FileNotFoundError:
        print(f"[ERRO] Arquivo '{caminho}' não encontrado. Verifique o nome e o diretório.")
        return []
    except PermissionError:
        print(f"[ERRO] Sem permissão de leitura para '{caminho}'.")
        return []

    return linhas

print("Funções definidas: ler_transacoes()")

Funções definidas: ler_transacoes()


In [108]:
# --- Teste rápido ---
brutas = ler_transacoes()
print(f"Linhas lidas: {len(brutas)}")
print(f"Colunas.....: {list(brutas[0].keys())}")
print("\nPrimeira linha:")
print(brutas[0])

print("\n--- Teste do FileNotFoundError (arquivo inexistente) ---")
resultado = ler_transacoes("arquivo_que_nao_existe.csv")
print(f"Retorno: {resultado} (o programa continuou rodando normalmente)")

Linhas lidas: 27
Colunas.....: ['id', 'data', 'cliente_id', 'tipo', 'valor', 'descricao', 'categoria']

Primeira linha:
{'id': '1', 'data': '2026-01-05', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': '3500.00', 'descricao': 'Salário janeiro', 'categoria': 'salario'}

--- Teste do FileNotFoundError (arquivo inexistente) ---
[ERRO] Arquivo 'arquivo_que_nao_existe.csv' não encontrado. Verifique o nome e o diretório.
Retorno: [] (o programa continuou rodando normalmente)


## Fase 5. Validação e limpeza

Cada linha passa por `validar_transacao()`, que devolve o registro limpo ou `None`.
Linhas inválidas são **descartadas silenciosamente**: nada de `raise`, o processamento continua.

Regras de descarte:

| Campo | Descarta quando |
|---|---|
| `id` | vazio ou não numérico |
| `cliente_id` | vazio |
| `data` | fora do formato AAAA-MM-DD |
| `tipo` | diferente de `credito` / `debito` |
| `valor` | não numérico ou menor ou igual a zero |

As conversões de tipo ficam isoladas em `converter_data()` e `converter_valor()`: **requisitos obrigatórios**. Dessa forma, ficam isoladas, reutilizáveis e testáveis separadamente.

Também tratamos os **registros duplicados** citados no cenário: `id` já processado é descartado e contabilizado à parte.

In [109]:
def converter_data(texto):
    """Converte texto AAAA-MM-DD em datetime. Retorna None se o formato for inválido."""
    try:
        return datetime.strptime(texto.strip(), FORMATO_DATA)
    except (ValueError, TypeError, AttributeError):
        return None


def converter_valor(texto):
    """Converte texto em float. Retorna None se não for um número válido."""
    try:
        return float(texto.strip())
    except (ValueError, TypeError, AttributeError):
        return None


def converter_id(texto):
    """Converte o id em inteiro. Retorna None se estiver vazio ou não for numérico."""
    try:
        return int(texto.strip())
    except (ValueError, TypeError, AttributeError):
        return None

print("Funções definidas: converter_data(), converter_valor(), converter_id()")

Funções definidas: converter_data(), converter_valor(), converter_id()


In [110]:
def validar_transacao(linha):
    """Valida uma única linha bruta do CSV.

    Retorna o registro limpo (dict) ou None se a linha for inválida.
    """
    try:
        id_bruto = linha["id"]
        data_bruta = linha["data"]
        cliente_bruto = linha["cliente_id"]
        tipo_bruto = linha["tipo"]
        valor_bruto = linha["valor"]
    except KeyError:
        # Cabeçalho fora do padrão esperado: descarta a linha.
        return None

    # id: obrigatório e numérico
    id_transacao = converter_id(id_bruto)
    if id_transacao is None:
        return None

    # cliente_id: não pode ser vazio
    cliente_id = (cliente_bruto or "").strip()
    if not cliente_id:
        return None

    # data: precisa estar em AAAA-MM-DD
    data = converter_data(data_bruta)
    if data is None:
        return None

    # tipo: apenas credito ou debito
    tipo = (tipo_bruto or "").strip().lower()
    if tipo not in TIPOS_VALIDOS:
        return None

    # valor: numérico e maior que zero
    valor = converter_valor(valor_bruto)
    if valor is None or valor <= 0:
        return None

    return {
        "id": id_transacao,
        "data": data,
        "mes": data.strftime(FORMATO_MES),
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": round(valor, 2),
        "descricao": (linha.get("descricao") or "").strip(),
        "categoria": (linha.get("categoria") or "").strip().lower(),
        "suspeita": valor > LIMITE_SUSPEITO,
    }

print("Funções definidas: validar_transacao()")

Funções definidas: validar_transacao()


In [111]:
def limpar_transacoes(linhas_brutas):
    """Aplica a validação a todas as linhas e devolve (válidas, estatísticas)."""
    validas = []
    ids_vistos = set()
    invalidas = 0
    duplicadas = 0

    for linha in linhas_brutas:
        transacao = validar_transacao(linha)

        if transacao is None:
            invalidas += 1
            continue

        if transacao["id"] in ids_vistos:
            duplicadas += 1
            continue

        ids_vistos.add(transacao["id"])
        validas.append(transacao)

    estatisticas = {
        "lidas": len(linhas_brutas),
        "validas": len(validas),
        "invalidas": invalidas,
        "duplicadas": duplicadas,
    }
    return validas, estatisticas

print("Funções definidas: limpar_transacoes()")

Funções definidas: limpar_transacoes()


In [112]:
# --- Teste rápido: conversores ---
print("converter_data:")
for teste in ["2026-01-05", "30-05-2026", "", "2026-13-45"]:
    print(f"  {teste!r:>14} -> {converter_data(teste)}")

print("\nconverter_valor:")
for teste in ["3500.00", "abc", "-75.00", ""]:
    print(f"  {teste!r:>14} -> {converter_valor(teste)}")

converter_data:
    '2026-01-05' -> 2026-01-05 00:00:00
    '30-05-2026' -> None
              '' -> None
    '2026-13-45' -> None

converter_valor:
       '3500.00' -> 3500.0
           'abc' -> None
        '-75.00' -> -75.0
              '' -> None


In [113]:
# --- Teste rápido: validação linha a linha ---
exemplos = [
    {"id": "1", "data": "2026-01-05", "cliente_id": "CLI001", "tipo": "credito",
     "valor": "3500.00", "descricao": "Salário", "categoria": "salario"},
    {"id": "21", "data": "2026-04-30", "cliente_id": "CLI001", "tipo": "debito",
     "valor": "abc", "descricao": "Valor inválido", "categoria": "compra"},
    {"id": "22", "data": "2026-05-03", "cliente_id": "", "tipo": "debito",
     "valor": "200.00", "descricao": "Sem cliente", "categoria": "transferencia"},
    {"id": "9", "data": "2026-02-14", "cliente_id": "CLI003", "tipo": "credito",
     "valor": "15000.00", "descricao": "Transferência", "categoria": "transferencia"},
]

for exemplo in exemplos:
    resultado = validar_transacao(exemplo)
    if resultado is None:
        print(f"id={exemplo['id']:>3} -> DESCARTADA")
    else:
        marcador = "  <-- SUSPEITA" if resultado["suspeita"] else ""
        print(f"id={resultado['id']:>3} -> ok | mês {resultado['mes']} | "
              f"{formatar_moeda(resultado['valor'])}{marcador}")

id=  1 -> ok | mês 2026-01 | R$ 3.500,00
id= 21 -> DESCARTADA
id= 22 -> DESCARTADA
id=  9 -> ok | mês 2026-02 | R$ 15.000,00  <-- SUSPEITA


In [114]:
# --- Teste rápido: limpeza completa ---
validas, estatisticas = limpar_transacoes(brutas)

print(f"Total de linhas lidas: {estatisticas['lidas']}")
print(f"Linhas válidas: {estatisticas['validas']}")
print(f"Linhas inválidas: {estatisticas['invalidas']}")
print(f"Linhas duplicadas descartadas: {estatisticas['duplicadas']}")

Total de linhas lidas: 27
Linhas válidas: 20
Linhas inválidas: 6
Linhas duplicadas descartadas: 1


## Fase 6. Agrupamento mensal, métricas e suspeitas

`gerar_relatorio()` percorre as transações válidas uma única vez, acumulando as métricas em um dicionário indexado pelo mês (`AAAA-MM`), conforme o padrão sugerido no enunciado do desafio.

Métricas por mês:
quantidade, total de crédito, total de débito, saldo (crédito − débito),
valor médio, maior e menor valor.

O bloco de período usa `min()` e `max()` sobre os objetos `datetime` e calcula a diferença em dias entre a transação mais antiga e a mais recente (requisito obrigatório).

In [115]:
def calcular_periodo(transacoes):
    """Devolve data inicial, final e a quantidade de dias entre elas."""
    datas = [t["data"] for t in transacoes]
    data_inicial = min(datas)
    data_final = max(datas)
    dias = (data_final - data_inicial).days

    return {
        "data_inicial": data_inicial.strftime(FORMATO_DATA),
        "data_final": data_final.strftime(FORMATO_DATA),
        "dias_entre": dias,
    }


def listar_suspeitas(transacoes):
    """Devolve as transações marcadas como suspeitas, das mais recentes para as mais antigas."""
    suspeitas = [t for t in transacoes if t["suspeita"]]
    suspeitas.sort(key=lambda t: t["data"], reverse=True)

    return [
        {
            "id": t["id"],
            "cliente_id": t["cliente_id"],
            "data": t["data"].strftime(FORMATO_DATA),
            "valor": t["valor"],
        }
        for t in suspeitas
    ]

print("Funções definidas: calcular_periodo(), listar_suspeitas()")

Funções definidas: calcular_periodo(), listar_suspeitas()


In [116]:
def gerar_relatorio(transacoes, estatisticas):
    """Agrupa as transações por mês e calcula todas as métricas do relatório."""
    resumo_mensal = {}

    for transacao in transacoes:
        mes = transacao["mes"]

        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "saldo": 0.0,
                "media": 0.0,
                "maior_valor": 0.0,
                "menor_valor": 0.0,
                "_valores": [],
            }

        dados_mes = resumo_mensal[mes]
        valor = transacao["valor"]

        dados_mes["quantidade"] += 1
        dados_mes["_valores"].append(valor)

        if transacao["tipo"] == "credito":
            dados_mes["total_credito"] += valor
        else:
            dados_mes["total_debito"] += valor

    # Fecha os cálculos que dependem do total acumulado do mês.
    for dados_mes in resumo_mensal.values():
        valores = dados_mes.pop("_valores")

        dados_mes["total_credito"] = round(dados_mes["total_credito"], 2)
        dados_mes["total_debito"] = round(dados_mes["total_debito"], 2)
        dados_mes["saldo"] = round(dados_mes["total_credito"] - dados_mes["total_debito"], 2)
        dados_mes["media"] = round(sum(valores) / len(valores), 2)
        dados_mes["maior_valor"] = round(max(valores), 2)
        dados_mes["menor_valor"] = round(min(valores), 2)

    # Ordena os meses cronologicamente.
    resumo_mensal = dict(sorted(resumo_mensal.items()))

    return {
        "gerado_em": date.today().strftime(FORMATO_DATA),
        "total_transacoes_validas": estatisticas["validas"],
        "total_transacoes_invalidas": estatisticas["invalidas"],
        "total_transacoes_duplicadas": estatisticas["duplicadas"],
        "total_linhas_lidas": estatisticas["lidas"],
        "limite_suspeito": LIMITE_SUSPEITO,
        "periodo": calcular_periodo(transacoes),
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": listar_suspeitas(transacoes),
    }

print("Funções definidas: gerar_relatorio()")

Funções definidas: gerar_relatorio()


In [117]:
# --- Teste rápido ----
relatorio = gerar_relatorio(validas, estatisticas)

print(f"Meses encontrados: {list(relatorio['resumo_mensal'].keys())}")
print(f"Período: {relatorio['periodo']['data_inicial']} -> "
      f"{relatorio['periodo']['data_final']} ({relatorio['periodo']['dias_entre']} dias)")
print(f"Suspeitas: {len(relatorio['transacoes_suspeitas'])}")
print("\nMétricas de 2026-01:")
for chave, valor in relatorio["resumo_mensal"]["2026-01"].items():
    print(f"  {chave:<15} {valor}")

Meses encontrados: ['2026-01', '2026-02', '2026-03', '2026-04']
Período: 2026-01-05 -> 2026-04-27 (112 dias)
Suspeitas: 3

Métricas de 2026-01:
  quantidade      6
  total_credito   20200.0
  total_debito    600.4
  saldo           19599.6
  media           3466.73
  maior_valor     12500.0
  menor_valor     99.9


## Fase 7. Exportação em JSON

`json.dump()` com `ensure_ascii=False` (para gravar acentos de verdade, e não `\u00e9`) e `indent=2` (para o arquivo ficar legível e versionável no Git).

In [118]:
def salvar_json(relatorio, caminho=ARQUIVO_SAIDA):
    """Salva o relatório em JSON formatado. Retorna True em caso de sucesso."""
    try:
        with open(caminho, "w", encoding="utf-8") as arquivo:
            json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)
    except (OSError, TypeError) as erro:
        print(f"[ERRO] Não foi possível salvar '{caminho}': {erro}")
        return False

    return True

print("Funções definidas: salvar_json()")

Funções definidas: salvar_json()


In [119]:
# --- Teste rápido ---
if salvar_json(relatorio):
    print(f"Arquivo '{ARQUIVO_SAIDA}' salvo com sucesso.\n")
    with open(ARQUIVO_SAIDA, "r", encoding="utf-8") as arquivo:
        conteudo = arquivo.read()
    print("Primeiras linhas do arquivo gerado:")
    print("\n".join(conteudo.splitlines()[:22]))

Arquivo 'relatorio.json' salvo com sucesso.

Primeiras linhas do arquivo gerado:
{
  "gerado_em": "2026-08-03",
  "total_transacoes_validas": 20,
  "total_transacoes_invalidas": 6,
  "total_transacoes_duplicadas": 1,
  "total_linhas_lidas": 27,
  "limite_suspeito": 10000.0,
  "periodo": {
    "data_inicial": "2026-01-05",
    "data_final": "2026-04-27",
    "dias_entre": 112
  },
  "resumo_mensal": {
    "2026-01": {
      "quantidade": 6,
      "total_credito": 20200.0,
      "total_debito": 600.4,
      "saldo": 19599.6,
      "media": 3466.73,
      "maior_valor": 12500.0,
      "menor_valor": 99.9
    },


## Fase 8. Exibição formatada no terminal

Saída dividida em três blocos: cabeçalho com o resumo da limpeza e o período analisado, relatório mensal e lista de transações suspeitas. Todos os valores monetários passam por `formatar_moeda()` para garantir a exibição da moeda com formatação correta.

In [120]:
LARGURA = 46


def exibir_cabecalho(relatorio):
    """Imprime o resumo da limpeza e o período analisado."""
    periodo = relatorio["periodo"]

    print("=" * LARGURA)
    print("        CLEARBANK - ANÁLISE FINANCEIRA")
    print("=" * LARGURA)
    print(f"Gerado em: {relatorio['gerado_em']}")
    print()
    print(f"Total de linhas lidas: {relatorio['total_linhas_lidas']}")
    print(f"Linhas válidas: {relatorio['total_transacoes_validas']}")
    print(f"Linhas inválidas: {relatorio['total_transacoes_invalidas']}")
    print(f"Linhas duplicadas descartadas: {relatorio['total_transacoes_duplicadas']}")
    print()
    print(f"Período analisado: {periodo['data_inicial']} -> {periodo['data_final']}")
    print(f"Intervalo: {periodo['dias_entre']} dias")


def exibir_resumo_mensal(relatorio):
    """Imprime as métricas de cada mês."""
    print()
    print("=" * LARGURA)
    print("           ===== RELATÓRIO MENSAL =====")
    print("=" * LARGURA)

    for mes, dados in relatorio["resumo_mensal"].items():
        print(f"\nMês: {mes}")
        print(f"  Transações:    {dados['quantidade']}")
        print(formatar_linha("Total crédito:", dados["total_credito"]))
        print(formatar_linha("Total débito:", dados["total_debito"]))
        print(formatar_linha("Saldo:", dados["saldo"]))
        print(formatar_linha("Média:", dados["media"]))
        print(formatar_linha("Maior valor:", dados["maior_valor"]))
        print(formatar_linha("Menor valor:", dados["menor_valor"]))


def exibir_suspeitas(relatorio):
    """Imprime as transações acima do limite suspeito."""
    suspeitas = relatorio["transacoes_suspeitas"]

    print()
    print("=" * LARGURA)
    print("        ===== TRANSAÇÕES SUSPEITAS =====")
    print("=" * LARGURA)
    print(f"Limite configurado: {formatar_moeda(relatorio['limite_suspeito'])}\n")

    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
        return

    for t in suspeitas:
        print(f"ID: {t['id']} | Cliente: {t['cliente_id']} | "
              f"Data: {t['data']} | Valor: {formatar_moeda(t['valor'])}")


def exibir_relatorio(relatorio):
    """Imprime o relatório completo, seção por seção."""
    exibir_cabecalho(relatorio)
    exibir_resumo_mensal(relatorio)
    exibir_suspeitas(relatorio)
    print()
    print("=" * LARGURA)

print("Funções definidas: exibir_cabecalho(), exibir_resumo_mensal(), exibir_suspeitas(), exibir_relatorio()")

Funções definidas: exibir_cabecalho(), exibir_resumo_mensal(), exibir_suspeitas(), exibir_relatorio()


## Fase 9. Célula de Execução Principal

Encadeia todo o fluxo: leitura -> limpeza -> métricas -> JSON -> exibição.

`main()` só orquestra as chamadas; nenhuma regra de negócio mora aqui. Se a leitura falhar (arquivo ausente ou sem nenhuma linha válida), o programa avisa e encerra de forma controlada.

In [121]:
def main():
    """Executa o pipeline completo da análise financeira."""
    brutas = ler_transacoes(ARQUIVO_ENTRADA)

    if not brutas:
        print("Nenhuma transação foi lida. Encerrando a análise.")
        return None

    validas, estatisticas = limpar_transacoes(brutas)

    if not validas:
        print("Nenhuma transação válida encontrada. Encerrando a análise.")
        return None

    relatorio = gerar_relatorio(validas, estatisticas)

    if salvar_json(relatorio, ARQUIVO_SAIDA):
        print(f"[OK] Relatório salvo em '{ARQUIVO_SAIDA}'.\n")

    exibir_relatorio(relatorio)
    return relatorio


relatorio_final = main()

print("Funções definidas: main()")

[OK] Relatório salvo em 'relatorio.json'.

        CLEARBANK - ANÁLISE FINANCEIRA
Gerado em: 2026-08-03

Total de linhas lidas: 27
Linhas válidas: 20
Linhas inválidas: 6
Linhas duplicadas descartadas: 1

Período analisado: 2026-01-05 -> 2026-04-27
Intervalo: 112 dias

           ===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações:    6
  Total crédito: R$ 20.200,00
  Total débito:  R$ 600,40
  Saldo:         R$ 19.599,60
  Média:         R$ 3.466,73
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 99,90

Mês: 2026-02
  Transações:    5
  Total crédito: R$ 18.500,00
  Total débito:  R$ 1.859,90
  Saldo:         R$ 16.640,10
  Média:         R$ 4.071,98
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 89,90

Mês: 2026-03
  Transações:    5
  Total crédito: R$ 7.700,00
  Total débito:  R$ 3.160,15
  Saldo:         R$ 4.539,85
  Média:         R$ 2.172,03
  Maior valor:   R$ 4.200,00
  Menor valor:   R$ 99,90

Mês: 2026-04
  Transações:    4
  Total crédito: R$ 15.300,00
  Total débi

# Requisitos Opcionais

## Análise com pandas

Mesmo agrupamento, agora com `pd.read_csv()` + `groupby`. Serve como **conferência cruzada**: se as duas implementações divergirem, uma das duas está errada.

A validação aqui é feita com máscaras booleanas em vez de laço — `to_numeric(errors="coerce")` e `to_datetime(errors="coerce")` transformam o que não converte em `NaN`/`NaT`, que então são filtrados.

A versão completa e executável em separado está no arquivo **`analise_pandas.py`** do repositório.

In [122]:
import pandas as pd

df = pd.read_csv(ARQUIVO_ENTRADA, dtype=str)

# Conversões: o que não converter vira NaN/NaT e é descartado no filtro seguinte.
df["id_num"] = pd.to_numeric(df["id"], errors="coerce")
df["valor_num"] = pd.to_numeric(df["valor"], errors="coerce")
df["data_dt"] = pd.to_datetime(df["data"], format=FORMATO_DATA, errors="coerce")
df["tipo_norm"] = df["tipo"].str.strip().str.lower()
df["cliente_norm"] = df["cliente_id"].fillna("").str.strip()

filtro_valido = (
    df["id_num"].notna()
    & df["valor_num"].notna()
    & (df["valor_num"] > 0)
    & df["data_dt"].notna()
    & df["tipo_norm"].isin(TIPOS_VALIDOS)
    & (df["cliente_norm"] != "")
)

df_valido = df[filtro_valido].drop_duplicates(subset="id_num", keep="first").copy()
df_valido["mes"] = df_valido["data_dt"].dt.strftime(FORMATO_MES)

# Colunas auxiliares para somar crédito e débito separadamente no groupby.
df_valido["credito"] = df_valido["valor_num"].where(df_valido["tipo_norm"] == "credito", 0.0)
df_valido["debito"] = df_valido["valor_num"].where(df_valido["tipo_norm"] == "debito", 0.0)

resumo_pandas = df_valido.groupby("mes").agg(
    quantidade=("valor_num", "size"),
    total_credito=("credito", "sum"),
    total_debito=("debito", "sum"),
    media=("valor_num", "mean"),
    maior_valor=("valor_num", "max"),
    menor_valor=("valor_num", "min"),
).round(2)

resumo_pandas["saldo"] = (resumo_pandas["total_credito"] - resumo_pandas["total_debito"]).round(2)
resumo_pandas = resumo_pandas[
    ["quantidade", "total_credito", "total_debito", "saldo", "media", "maior_valor", "menor_valor"]
]

print(f"Linhas lidas...: {len(df)}")
print(f"Linhas válidas.: {len(df_valido)}")
print()
resumo_pandas

Linhas lidas...: 27
Linhas válidas.: 20



,quantidade,total_credito,total_debito,saldo,media,maior_valor,menor_valor
mes,,,,,,,
2026-01,6,20200.0,600.40,19599.60,3466.73,12500.0,99.90
2026-02,5,18500.0,1859.90,16640.10,4071.98,15000.0,89.90
2026-03,5,7700.0,3160.15,4539.85,2172.03,4200.0,99.90
2026-04,4,15300.0,1990.75,13309.25,4322.69,11800.0,540.75


### Conferência: nativa × pandas

Comparação métrica a métrica. Qualquer divergência acima de um centavo é sinalizada.

In [123]:
divergencias = 0

print(f"{'Mês':<10}{'Métrica':<16}{'Nativo':>14}{'Pandas':>14}   Status")
print("-" * 62)

for mes, metricas_nativas in relatorio_final["resumo_mensal"].items():
    for chave, valor_nativo in metricas_nativas.items():
        valor_pandas = resumo_pandas.loc[mes, chave]
        igual = abs(float(valor_nativo) - float(valor_pandas)) < 0.01
        if not igual:
            divergencias += 1
        status = "OK" if igual else "DIVERGÊNCIA"
        print(f"{mes:<10}{chave:<16}{valor_nativo:>14}{valor_pandas:>14}   {status}")

print("-" * 62)
if divergencias == 0:
    print("Todas as métricas conferem entre a solução nativa e a versão com pandas.")
else:
    print(f"{divergencias} divergência(s) encontrada(s).")

Mês       Métrica                 Nativo        Pandas   Status
--------------------------------------------------------------
2026-01   quantidade                   6             6   OK
2026-01   total_credito          20200.0       20200.0   OK
2026-01   total_debito             600.4         600.4   OK
2026-01   saldo                  19599.6       19599.6   OK
2026-01   media                  3466.73       3466.73   OK
2026-01   maior_valor            12500.0       12500.0   OK
2026-01   menor_valor               99.9          99.9   OK
2026-02   quantidade                   5             5   OK
2026-02   total_credito          18500.0       18500.0   OK
2026-02   total_debito            1859.9        1859.9   OK
2026-02   saldo                  16640.1       16640.1   OK
2026-02   media                  4071.98       4071.98   OK
2026-02   maior_valor            15000.0       15000.0   OK
2026-02   menor_valor               89.9          89.9   OK
2026-03   quantidade             

## Visualização com matplotlib

Gráfico de **barras agrupadas** com crédito e débito por mês, somado a uma **linha do saldo** no mesmo eixo, assim dá para ler o volume movimentado e o resultado líquido de uma vez só.

Inclui título, rótulos nos dois eixos, legenda e grade horizontal. Salvo como `grafico.png`.

In [124]:
import matplotlib
matplotlib.use("Agg")  # backend sem interface, garante o salvamento em qualquer ambiente
import matplotlib.pyplot as plt

meses = list(relatorio_final["resumo_mensal"].keys())
creditos = [relatorio_final["resumo_mensal"][m]["total_credito"] for m in meses]
debitos = [relatorio_final["resumo_mensal"][m]["total_debito"] for m in meses]
saldos = [relatorio_final["resumo_mensal"][m]["saldo"] for m in meses]

posicoes = range(len(meses))
largura = 0.38

figura, eixo = plt.subplots(figsize=(10, 5.5))

eixo.bar([p - largura / 2 for p in posicoes], creditos, largura,
         label="Crédito", color="#2E7D32")
eixo.bar([p + largura / 2 for p in posicoes], debitos, largura,
         label="Débito", color="#C62828")
eixo.plot(posicoes, saldos, marker="o", linewidth=2, color="#1565C0",
          label="Saldo (crédito - débito)")

eixo.axhline(0, color="#555555", linewidth=0.8)
eixo.set_title("ClearBank - Crédito, Débito e Saldo por Mês", fontsize=14, fontweight="bold")
eixo.set_xlabel("Mês de referência (AAAA-MM)")
eixo.set_ylabel("Valor (R$)")
eixo.set_xticks(list(posicoes))
eixo.set_xticklabels(meses)
eixo.legend()
eixo.grid(axis="y", linestyle="--", alpha=0.4)

# Eixo Y no padrão brasileiro de milhar.
eixo.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", "."))
)

plt.tight_layout()
plt.savefig(ARQUIVO_GRAFICO, dpi=150)
print(f"Gráfico salvo em '{ARQUIVO_GRAFICO}'.")
plt.show()

Gráfico salvo em 'grafico.png'.


## Arquivos gerados

Conferência final do que foi produzido pela execução do notebook.

In [125]:
for nome in [ARQUIVO_ENTRADA, ARQUIVO_SAIDA, ARQUIVO_GRAFICO]:
    if os.path.exists(nome):
        tamanho = os.path.getsize(nome)
        print(f"[OK] {nome:<20} {tamanho:>8} bytes")
    else:
        print(f"[--] {nome:<20} não encontrado")

[OK] transacoes.csv           1638 bytes
[OK] relatorio.json           1541 bytes
[OK] grafico.png             80990 bytes
